# NB17 — Reviewer-requested sensitivity analyses (CPU)

**Runtime:** CPU. No GPU is needed: SVM, random forest, PLS-DA and ordinal logistic regression (mord) are CPU-only.

Three analyses on the frozen 17,662-observation master file with the candidate grids of manuscript Table 2:

| Part | Question | Approx. time (Colab CPU) |
|---|---|---|
| **A** | Positive control: pooled stratified random 5-fold CV and within-experiment 5-fold CV versus nested LOEO (TPH, Spectral, TPH + Spectral) | 2–4 h |
| **B** | MSC versus RAW and SNV under nested LOEO (Spectral) | 1–1.5 h |
| **C** | Calibrators: temperature, vector scaling and one-vs-rest isotonic regression (Spectral RAW) | 15–30 min |

Each part checkpoints its outputs after every modality or arm, so a disconnected session can be resumed by re-running from cell 1. The last cell compares the LOEO arm with manuscript Table 4 and zips the outputs.

In [ ]:
# 1. Mount Google Drive and install the frozen package versions
from google.colab import drive
drive.mount('/content/drive')
!pip -q install mord==0.7 xgboost==3.4.1

In [ ]:
# 2. Paths and columns (from the project data dictionary). Edit PROJECT if your copy lives elsewhere.
import glob, os, time
PROJECT = '/content/drive/MyDrive/Paper3_Banano/BANANA_RIPENESS_MULTISENSOR_Q1'
MASTER_CSV = f'{PROJECT}/02_EXTRACTED_DATA/06_master_original_multisensor.csv'
if not os.path.exists(MASTER_CSV):   # fall back to a Drive-wide search
    hits = glob.glob('/content/drive/MyDrive/**/06_master_original_multisensor.csv', recursive=True)
    assert hits, '06_master_original_multisensor.csv not found'
    MASTER_CSV = hits[0]; PROJECT = MASTER_CSV.split('/02_EXTRACTED_DATA')[0]
OUT_DIR = f'{PROJECT}/06_RESULTS/NB17_BLINDAJE_REVISORES'
os.makedirs(OUT_DIR, exist_ok=True)
TEST_COL, STAGE_COL = 'test_id', 'stage'
TPH_COLS = ['Temp-int', 'Press-int', 'Humid-int']
SPEC_COLS = ['SpA410', 'SpB435', 'SpC460', 'SpD485', 'SpE510', 'SpF535', 'SpG560', 'SpH585', 'SpR610',
             'SpI645', 'SpS680', 'SpJ705', 'SpT730', 'SpU760', 'SpV810', 'SpW860', 'SpK900', 'SpL940']
print('Master:', MASTER_CSV); print('Outputs:', OUT_DIR)

In [ ]:
# 3. Functions (Table 2 models, nested LOEO, random-split control, MSC, calibrators)
import os, re, json, time, warnings
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from scipy.optimize import minimize_scalar
warnings.filterwarnings('ignore')

SEED = 2026
WAVELENGTHS = [410, 435, 460, 485, 510, 535, 560, 585, 610, 645, 680, 705, 730, 760, 810, 860, 900, 940]

try:
    import mord
    ORDINAL = lambda a: mord.LogisticAT(alpha=a)
except ImportError:  # only for code testing; the manuscript analysis requires mord
    raise ImportError('mord is required: pip install mord==0.7')
from xgboost import XGBClassifier

# ----------------------------------------------------------------------------- data
def load():
    df = pd.read_csv(MASTER_CSV)
    df = df[df[TEST_COL].isin([2, 3, 4, 5])].reset_index(drop=True)
    df[TEST_COL] = df[TEST_COL].astype(int); df[STAGE_COL] = df[STAGE_COL].astype(int)
    assert df[TPH_COLS + SPEC_COLS].notna().all().all(), 'missing values in TPH/spectral columns'
    return df

# ----------------------------------------------------------------------------- models (Table 2)
class PLSDA(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=2): self.n_components = n_components
    def fit(self, X, y):
        self.classes_ = np.unique(y); Y = (y[:, None] == self.classes_[None, :]).astype(float)
        self.pls_ = PLSRegression(n_components=min(self.n_components, X.shape[1]), scale=False).fit(X, Y); return self
    def decision_function(self, X): return self.pls_.predict(X)
    def predict(self, X): return self.classes_[np.argmax(self.decision_function(X), 1)]
    def predict_proba(self, X):
        s = self.decision_function(X); e = np.exp(s - s.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

class XGBWrap(BaseEstimator, ClassifierMixin):
    """XGBoost on labels re-indexed to 0..K-1 within each training partition."""
    def __init__(self, n_estimators=250, max_depth=3, learning_rate=0.05):
        self.n_estimators, self.max_depth, self.learning_rate = n_estimators, max_depth, learning_rate
    def fit(self, X, y):
        self.classes_ = np.unique(y); yi = np.searchsorted(self.classes_, y)
        self.m_ = XGBClassifier(n_estimators=self.n_estimators, max_depth=self.max_depth, learning_rate=self.learning_rate,
                                subsample=0.9, colsample_bytree=0.9, random_state=SEED, n_jobs=-1, verbosity=0,
                                objective='multi:softprob' if len(self.classes_) > 2 else 'binary:logistic').fit(X, yi)
        return self
    def predict_proba(self, X):
        p = self.m_.predict_proba(X); return p if p.ndim == 2 else np.c_[1 - p, p]
    def predict(self, X): return self.classes_[np.argmax(self.predict_proba(X), 1)]

class SVMWrap(BaseEstimator, ClassifierMixin):
    def __init__(self, C=2.0, gamma='scale'): self.C, self.gamma = C, gamma
    def fit(self, X, y): self.m_ = SVC(C=self.C, gamma=self.gamma, random_state=SEED).fit(X, y); self.classes_ = self.m_.classes_; return self
    def decision_function(self, X):
        d = self.m_.decision_function(X); return d if d.ndim == 2 else np.c_[-d, d]
    def predict(self, X): return self.m_.predict(X)
    def predict_proba(self, X):   # softmax surrogate, as in the manuscript
        s = self.decision_function(X); e = np.exp(s - s.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

def candidates(modality):
    nc = [1, 2, 3] if modality == 'TPH' else [2, 4, 8]
    C = []
    C += [('OrdinalLogistic', a, lambda a=a: make_pipeline(StandardScaler(), ORDINAL(a))) for a in (0.01, 0.1, 1, 10)]
    C += [('PLSDA', k, lambda k=k: make_pipeline(StandardScaler(), PLSDA(k))) for k in nc]
    C += [('SVM-RBF', (c, g), lambda c=c, g=g: make_pipeline(StandardScaler(), SVMWrap(c, g))) for c, g in ((0.5, 'scale'), (2, 'scale'), (8, 'scale'), (8, 0.03))]
    C += [('RandomForest', (d, l, f), lambda d=d, l=l, f=f: RandomForestClassifier(400, max_depth=d, min_samples_leaf=l, max_features=f, random_state=SEED, n_jobs=-1))
          for d in (None, 18) for l in (1, 3) for f in ('sqrt', 0.7)]
    C += [('XGBoost', (n, d, lr), lambda n=n, d=d, lr=lr: XGBWrap(n, d, lr)) for n, d, lr in ((250, 2, 0.10), (250, 3, 0.05), (500, 4, 0.03), (500, 5, 0.05))]
    return C

def macro_f1(y, p): return f1_score(y, p, labels=np.unique(y), average='macro', zero_division=0)

# ----------------------------------------------------------------------------- preprocessing (fitted on training rows only)
def snv(X): return (X - X.mean(1, keepdims=True)) / (X.std(1, ddof=1, keepdims=True) + 1e-12)
class MSC:
    def fit(self, X): self.ref_ = X.mean(0); return self
    def transform(self, X):
        out = np.empty_like(X, dtype=float)
        for i, x in enumerate(X):
            b, a = np.polyfit(self.ref_, x, 1); out[i] = (x - a) / (b if abs(b) > 1e-12 else 1e-12)
        return out
def prep_pair(kind, Xtr, Xte):
    if kind == 'RAW': return Xtr, Xte
    if kind == 'SNV': return snv(Xtr), snv(Xte)
    if kind == 'MSC': m = MSC().fit(Xtr); return m.transform(Xtr), m.transform(Xte)
    raise ValueError(kind)

# ----------------------------------------------------------------------------- nested LOEO (as NB03/NB04)
def select(X, y, g, modality, prep='RAW'):
    """Inner experiment-disjoint selection on development data; returns (family, params, factory)."""
    tests = np.unique(g); scores = {}
    for fam, par, fac in candidates(modality):
        s = []
        for v in tests:
            tr, va = g != v, g == v
            Xtr, Xva = prep_pair(prep, X[tr], X[va])
            s.append(macro_f1(y[va], fac().fit(Xtr, y[tr]).predict(Xva)))
        scores[(fam, par)] = (np.mean(s), fac)
    best_per_family = {}
    for (fam, par), (sc, fac) in scores.items():
        if fam not in best_per_family or sc > best_per_family[fam][0]: best_per_family[fam] = (sc, par, fac)
    fam = max(best_per_family, key=lambda f: best_per_family[f][0])
    return fam, best_per_family[fam][1], best_per_family[fam][2], best_per_family[fam][0]

def nested_loeo(df, feats, modality, tcol, scol, prep='RAW', keep_probs=False):
    X, y, g = df[feats].to_numpy(float), df[scol].to_numpy(int), df[tcol].to_numpy(int)
    rows, store = [], {}
    for t in sorted(np.unique(g)):
        dev, ext = g != t, g == t
        fam, par, fac, inner = select(X[dev], y[dev], g[dev], modality, prep)
        Xd, Xe = prep_pair(prep, X[dev], X[ext]); m = fac().fit(Xd, y[dev]); pred = m.predict(Xe)
        rows.append(dict(arm=f'LOEO_{prep}', modality=modality, outer_test=t, model=fam, params=str(par), inner_macro_f1=inner,
                         macro_f1=macro_f1(y[ext], pred), accuracy=accuracy_score(y[ext], pred), n=int(ext.sum())))
        if keep_probs: store[t] = dict(fam=fam, par=par, fac=fac, dev=dev, ext=ext)
        print(f'  LOEO {prep:3s} {modality:14s} Test {t}: {fam:15s} Macro-F1 {rows[-1]["macro_f1"]:.3f}')
    return pd.DataFrame(rows), store

# ----------------------------------------------------------------------------- A. positive control
def random_split(df, feats, modality, tcol, scol):
    X, y = df[feats].to_numpy(float), df[scol].to_numpy(int)
    rows = []
    # (i) pooled stratified 5-fold with nested random 3-fold selection
    outer = StratifiedKFold(5, shuffle=True, random_state=SEED)
    for k, (tr, te) in enumerate(outer.split(X, y)):
        inner_g = np.zeros(len(tr), int)
        for j, (_, va) in enumerate(StratifiedKFold(3, shuffle=True, random_state=SEED).split(X[tr], y[tr])): inner_g[va] = j
        fam, par, fac, inner = select(X[tr], y[tr], inner_g, modality)
        pred = fac().fit(X[tr], y[tr]).predict(X[te])
        rows.append(dict(arm='RANDOM_POOLED_5FOLD', modality=modality, outer_test=f'fold{k+1}', model=fam, params=str(par),
                         inner_macro_f1=inner, macro_f1=macro_f1(y[te], pred), accuracy=accuracy_score(y[te], pred), n=len(te)))
    # (ii) within-experiment stratified 5-fold (train and test from the same experiment), model fixed to the pooled winner
    fam_counts = pd.Series([r['model'] for r in rows]).value_counts(); fam = fam_counts.index[0]
    par = next(r['params'] for r in rows if r['model'] == fam)
    fac = next(f for fm, p, f in candidates(modality) if fm == fam and str(p) == par)
    for t in sorted(df[tcol].unique()):
        sub = (df[tcol] == t).to_numpy(); Xs, ys = X[sub], y[sub]
        keep = np.isin(ys, pd.Series(ys).value_counts()[lambda s: s >= 5].index)   # stages with >=5 rows
        Xs, ys = Xs[keep], ys[keep]; f1s, accs = [], []
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xs, ys):
            pred = fac().fit(Xs[tr], ys[tr]).predict(Xs[te]); f1s.append(macro_f1(ys[te], pred)); accs.append(accuracy_score(ys[te], pred))
        rows.append(dict(arm='WITHIN_EXPERIMENT_5FOLD', modality=modality, outer_test=t, model=fam, params=par, inner_macro_f1=np.nan,
                         macro_f1=np.mean(f1s), accuracy=np.mean(accs), n=int(len(ys))))
    print(f'  RANDOM {modality:14s} pooled Macro-F1 {np.mean([r["macro_f1"] for r in rows[:5]]):.3f}')
    return pd.DataFrame(rows)

# ----------------------------------------------------------------------------- C. calibrators
def ece(p, y_idx, bins=15):
    conf, pred = p.max(1), p.argmax(1); e = 0.0
    for lo, hi in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == y_idx[m]).mean() - conf[m].mean())
    return e
def scores(p, y_idx):
    p = np.clip(p, 1e-12, 1); p = p / p.sum(1, keepdims=True); oh = np.eye(p.shape[1])[y_idx]
    return dict(ECE15=ece(p, y_idx), NLL=-np.mean(np.log(p[np.arange(len(y_idx)), y_idx])), Brier=np.mean(((p - oh) ** 2).sum(1)),
                accuracy=np.mean(p.argmax(1) == y_idx))
def fit_temperature(logp, y_idx):
    f = lambda lt: -np.mean(np.log(np.clip(np.exp(logp / np.exp(lt) - np.logaddexp.reduce(logp / np.exp(lt), 1, keepdims=True))[np.arange(len(y_idx)), y_idx], 1e-12, 1)))
    T = np.exp(minimize_scalar(f, bounds=(np.log(1e-3), np.log(1e3)), method='bounded').x)
    return lambda lp: np.exp(lp / T - np.logaddexp.reduce(lp / T, 1, keepdims=True)), T
def fit_vector(logp, y_idx):
    lr = LogisticRegression(C=1e4, max_iter=5000).fit(logp, y_idx)            # per-class weights + biases on log-probabilities
    K = logp.shape[1]
    def apply(lp):
        q = np.zeros((len(lp), K)); q[:, lr.classes_] = lr.predict_proba(lp); return q
    return apply
def fit_isotonic(p, y_idx):
    K = p.shape[1]; irs = [IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=1).fit(p[:, k], (y_idx == k).astype(float)) for k in range(K)]
    def apply(q):
        r = np.column_stack([irs[k].predict(q[:, k]) for k in range(K)]) + 1e-6; return r / r.sum(1, keepdims=True)
    return apply

def calibrators(df, feats, tcol, scol, store):
    X, y, g = df[feats].to_numpy(float), df[scol].to_numpy(int), df[tcol].to_numpy(int)
    classes = np.unique(y); rows = []
    for t, s in store.items():
        dev_idx = np.where(s['dev'])[0]; gd = g[dev_idx]; oof = np.zeros((len(dev_idx), len(classes)))
        for v in np.unique(gd):                                   # cross-fitted development probabilities
            tr, va = dev_idx[gd != v], dev_idx[gd == v]
            m = s['fac']().fit(X[tr], y[tr]); pr = m.predict_proba(X[va]); cl = m.classes_ if hasattr(m, 'classes_') else m[-1].classes_
            oof[np.searchsorted(dev_idx, va)[:, None], np.searchsorted(classes, cl)[None, :]] = pr
        m = s['fac']().fit(X[dev_idx], y[dev_idx]); cl = m.classes_ if hasattr(m, 'classes_') else m[-1].classes_
        pe = np.zeros((int(s['ext'].sum()), len(classes))); pe[:, np.searchsorted(classes, cl)] = m.predict_proba(X[s['ext']])
        yd, ye = np.searchsorted(classes, y[dev_idx]), np.searchsorted(classes, y[s['ext']])
        lo_d, lo_e = np.log(np.clip(oof, 1e-12, 1)), np.log(np.clip(pe, 1e-12, 1))
        temp, T = fit_temperature(lo_d, yd)
        for name, q in [('uncalibrated', pe), ('temperature', temp(lo_e)), ('vector_scaling', fit_vector(lo_d, yd)(lo_e)),
                        ('isotonic_ovr', fit_isotonic(np.clip(oof, 1e-12, 1), yd)(pe))]:
            rows.append(dict(outer_test=t, model=s['fam'], calibrator=name, T=T if name == 'temperature' else np.nan, **scores(q, ye)))
        print(f'  CAL Test {t}: done (T = {T:.2f})')
    return pd.DataFrame(rows)



In [ ]:
# 4. Load and verify against manuscript Table 1
df = load()
counts = df.groupby(TEST_COL)[STAGE_COL].value_counts().unstack(fill_value=0).reindex(columns=range(1, 6), fill_value=0)
print(counts)
assert len(df) == 17662, len(df)
assert df[TEST_COL].value_counts().sort_index().tolist() == [1720, 4777, 6133, 5032]
assert counts.loc[2].tolist() == [186, 17, 475, 1042, 0] and counts.loc[5].tolist() == [2, 184, 1262, 2428, 1156]
MOD = {'TPH': TPH_COLS, 'Spectral': SPEC_COLS, 'TPH + Spectral': TPH_COLS + SPEC_COLS}
print('OK: matches manuscript Table 1')

In [ ]:
# 5. PART A — positive control (checkpointed per modality)
import pandas as pd
t0 = time.time()
for mod, feats in MOD.items():
    f = f"{OUT_DIR}/A_{mod.replace(' + ', '_').replace(' ', '')}.csv"
    if os.path.exists(f): print('already computed:', mod); continue
    r = pd.concat([random_split(df, feats, mod, TEST_COL, STAGE_COL), nested_loeo(df, feats, mod, TEST_COL, STAGE_COL)[0]])
    r.to_csv(f, index=False); print(f'{mod} done ({(time.time() - t0) / 60:.0f} min)')
A = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{OUT_DIR}/A_*.csv')) if 'summary' not in f])
summ = A.groupby(['modality', 'arm'])[['macro_f1', 'accuracy']].mean().round(3)
summ.to_csv(f'{OUT_DIR}/A_summary.csv'); summ

In [ ]:
# 6. PART B — MSC versus RAW and SNV (Spectral, nested LOEO)
t0 = time.time(); store = {}
for prep in ('RAW', 'SNV', 'MSC'):
    f = f'{OUT_DIR}/B_{prep}.csv'
    if os.path.exists(f): print('already computed:', prep); continue
    r, _ = nested_loeo(df, SPEC_COLS, 'Spectral', TEST_COL, STAGE_COL, prep)
    r.to_csv(f, index=False); print(f'{prep} done ({(time.time() - t0) / 60:.0f} min)')
B = pd.concat([pd.read_csv(f'{OUT_DIR}/B_{p}.csv') for p in ('RAW', 'SNV', 'MSC')])
B.pivot(index='outer_test', columns='arm', values='macro_f1').round(3)

In [ ]:
# 7. PART C — alternative calibrators (LOEO-selected Spectral RAW model)
f = f'{OUT_DIR}/C_calibrators.csv'
if os.path.exists(f):
    C = pd.read_csv(f)
else:
    _, store = nested_loeo(df, SPEC_COLS, 'Spectral', TEST_COL, STAGE_COL, 'RAW', keep_probs=True)
    C = calibrators(df, SPEC_COLS, TEST_COL, STAGE_COL, store); C.to_csv(f, index=False)
C.groupby('calibrator')[['ECE15', 'NLL', 'Brier', 'accuracy']].mean().round(3)

In [ ]:
# 8. Comparison with manuscript Table 4 (LOEO RAW) and output archive
T4 = {('Spectral', 2): 0.003, ('Spectral', 3): 0.230, ('Spectral', 4): 0.289, ('Spectral', 5): 0.000,
      ('TPH', 2): 0.167, ('TPH', 3): 0.093, ('TPH', 4): 0.131, ('TPH', 5): 0.193,
      ('TPH + Spectral', 2): 0.001, ('TPH + Spectral', 3): 0.217, ('TPH + Spectral', 4): 0.213, ('TPH + Spectral', 5): 0.000}
L = A[A.arm == 'LOEO_RAW'].copy(); L['outer_test'] = L.outer_test.astype(int)
L['table4_macro_f1'] = [T4[(m, t)] for m, t in zip(L.modality, L.outer_test)]
L[['modality', 'outer_test', 'model', 'macro_f1', 'table4_macro_f1']].to_csv(f'{OUT_DIR}/A_check_vs_table4.csv', index=False)
print(L[['modality', 'outer_test', 'model', 'macro_f1', 'table4_macro_f1']].round(3).to_string(index=False))
import json, shutil
json.dump({'seed': SEED, 'master': MASTER_CSV, 'n': len(df)}, open(f'{OUT_DIR}/run_manifest.json', 'w'), indent=1)
zip_path = shutil.make_archive(f'{PROJECT}/06_RESULTS/NB17_BLINDAJE_REVISORES_RESULTADOS', 'zip', OUT_DIR)
print('Archive:', zip_path)